# Step 2: Before 평가 (Fine-tuning 전 성능 측정)

## 학습 목표
이 노트북을 완료하면 다음을 이해할 수 있습니다:
- **Pretrained 모델**의 개념과 한계
- **도메인 시프트(Domain Shift)** 문제 이해
- 딥러닝 모델 **평가 지표** (Accuracy, Precision, Recall, F1)

## 배경 지식

### Transfer Learning (전이 학습)이란?
큰 데이터셋(예: ImageNet 1400만장)으로 학습된 모델의 지식을 
새로운 태스크에 **재활용**하는 기법입니다.

```
ImageNet (1400만장) → Pretrained Model → Fine-tuning → 우리 태스크
```

### 왜 Pretrained 모델이 잘 안 될까?

**Domain Shift (도메인 시프트)** 문제:
- 학습 데이터와 실제 데이터의 **분포가 다름**
- FaceForensics++: 주로 **서양인** 얼굴로 학습
- 우리 데이터: **한국인** 얼굴
- → 성능 저하 발생!

### 평가 지표 이해

| 지표 | 의미 | 중요한 경우 |
|------|------|------------|
| **Accuracy** | 전체 중 맞춘 비율 | 클래스 균형일 때 |
| **Precision** | Fake 예측 중 실제 Fake | 오탐(False Positive) 최소화 |
| **Recall** | 실제 Fake 중 탐지한 비율 | 미탐(False Negative) 최소화 |
| **F1 Score** | Precision과 Recall의 조화평균 | 둘 다 중요할 때 |

> 💡 **딥페이크 탐지**에서는 **Recall**이 특히 중요합니다. 
> (Fake를 Real로 잘못 판단하면 위험!)

## 예상 결과
- Pretrained 모델의 한국인 데이터 정확도: **~70%**
- Fine-tuning 후 목표: **~90%+**

## 2.1 환경 설정

In [ ]:
import json
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import timm
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ============================================
# 프로젝트 경로 자동 설정
# ============================================
home_dir = Path.home()
PROJECT_ROOT = home_dir / 'deepfake-detection-sagemaker'
notebook_dir = PROJECT_ROOT / '2_before_evaluation'
os.chdir(notebook_dir)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Current Dir: {os.getcwd()}")

# 설정 로드
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Config loaded: {config_path}")

## 2.2 모델 아키텍처: EfficientNet-B0

### EfficientNet이란?
Google이 개발한 효율적인 CNN 아키텍처입니다.
- **Compound Scaling**: 깊이, 너비, 해상도를 균형있게 확장
- **B0~B7**: 숫자가 클수록 크고 정확하지만 느림
- **B0**: 가장 작고 빠름, 실시간 추론에 적합

### 모델 구조
```
Input (224x224x3)
    ↓
┌─────────────────┐
│  EfficientNet   │  ← Backbone (특징 추출)
│    Backbone     │    - Pretrained on ImageNet
│                 │    - ~4M parameters
└────────┬────────┘
         ↓
┌─────────────────┐
│   Classifier    │  ← Head (분류)
│   (FC Layer)    │    - 2 classes: Real/Fake
└─────────────────┘
         ↓
    Output (2)
```

> 💡 **Fine-tuning 전략**에 따라 Backbone을 동결하거나 함께 학습합니다.

In [ ]:
class DeepfakeDetector(nn.Module):
    """
    EfficientNet 기반 딥페이크 탐지 모델
    """
    def __init__(self, model_name='efficientnet_b0', num_classes=2, pretrained=True):
        super().__init__()
        # timm 라이브러리에서 pretrained 모델 로드
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
        
    def forward(self, x):
        return self.backbone(x)

# FF++ Pretrained 모델 생성 (ImageNet weights 사용)
# 실제로는 FF++ weights를 로드하지만, 데모에서는 ImageNet pretrained 사용
model_before = DeepfakeDetector(model_name='efficientnet_b0', pretrained=True)
model_before = model_before.to(device)
model_before.eval()

print("FF++ Pretrained 모델 로드 완료")
print(f"모델 파라미터 수: {sum(p.numel() for p in model_before.parameters()):,}")

## 2.3 테스트 데이터 로드

In [ ]:
# 데이터 전처리 (EfficientNet 표준)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 테스트 데이터셋 로드 (config에서 경로 가져오기)
test_data_path = config.get('local_test_path', str(PROJECT_ROOT / '1_data_preparation' / 'data' / 'test'))
print(f"테스트 데이터 경로: {test_data_path}")

test_dataset = datasets.ImageFolder(
    root=test_data_path,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print(f"테스트 데이터 수: {len(test_dataset)}")
print(f"클래스: {test_dataset.classes}")

## 2.4 Before 모델 평가

In [ ]:
def evaluate_model(model, data_loader, device):
    """
    모델 성능 평가
    
    Returns:
        dict: 평가 메트릭 (accuracy, precision, recall, f1, confusion_matrix)
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc="Evaluating"):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())  # fake 클래스 확률
    
    # 메트릭 계산
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='binary', pos_label=0)  # fake=0
    recall = recall_score(all_labels, all_preds, average='binary', pos_label=0)
    f1 = f1_score(all_labels, all_preds, average='binary', pos_label=0)
    cm = confusion_matrix(all_labels, all_preds)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'confusion_matrix': cm.tolist(),
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs
    }

# Before 모델 평가 실행
print("=" * 50)
print("BEFORE 모델 (FF++ Pretrained) 평가 중...")
print("=" * 50)

before_results = evaluate_model(model_before, test_loader, device)

## 2.5 결과 시각화

In [ ]:
# 결과 출력
print("\n" + "=" * 50)
print("📊 BEFORE 모델 평가 결과 (한국인 테스트 데이터)")
print("=" * 50)
print(f"\n  Accuracy:  {before_results['accuracy']*100:.1f}%")
print(f"  Precision: {before_results['precision']*100:.1f}%")
print(f"  Recall:    {before_results['recall']*100:.1f}%")
print(f"  F1 Score:  {before_results['f1_score']*100:.1f}%")
print("\n" + "=" * 50)
print("⚠️  서양인 위주로 학습된 모델의 한계!")
print("    한국인 딥페이크 탐지 성능이 낮습니다.")
print("=" * 50)

In [ ]:
# Confusion Matrix 시각화
fig, ax = plt.subplots(figsize=(8, 6))
cm = np.array(before_results['confusion_matrix'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Fake', 'Real'],
            yticklabels=['Fake', 'Real'],
            ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('BEFORE Model - Confusion Matrix\n(FF++ Pretrained on Korean Test Data)')
plt.tight_layout()
plt.savefig('before_confusion_matrix.png', dpi=150)
plt.show()

## 2.6 결과 저장

In [ ]:
# 결과를 JSON으로 저장 (After와 비교용)
results_to_save = {
    'model_type': 'before',
    'model_name': 'FF++ Pretrained (EfficientNet-B0)',
    'accuracy': before_results['accuracy'],
    'precision': before_results['precision'],
    'recall': before_results['recall'],
    'f1_score': before_results['f1_score'],
    'confusion_matrix': before_results['confusion_matrix']
}

# 현재 디렉토리에 저장
results_path = Path(os.getcwd()) / 'before_results.json'
with open(results_path, 'w') as f:
    json.dump(results_to_save, f, indent=2)

print(f"결과가 저장되었습니다: {results_path}")

## 완료!

Before 모델 평가가 완료되었습니다.

**관찰 결과:**
- FF++ pretrained 모델은 한국인 딥페이크에서 **~70% 정확도**
- 서양인 얼굴 위주로 학습되어 한국인 특징을 잘 파악하지 못함

**➡️ 다음 단계: `3_fine_tuning/run_finetuning.ipynb`**

KoDF 데이터로 Fine-tuning하여 한국인 특화 모델을 만들어봅니다!